# Correction du TP11 du cours d'introduction à l'algorithmique

En cas de question, me contacter à l'adresse antoine.gauquier@ens.fr

## Rappel du code de Graphe et Sommet

In [46]:
class Sommet:
    def __init__(self, id, label):
        self.id = id
        self.label = label
        self.voisins = []

    def ajout_voisin(self, nouveau_voisin):
        if nouveau_voisin not in self.voisins:
            self.voisins.append(nouveau_voisin)

class Graphe:
    def __init__(self):
        self.sommets = {}
        self.n = 0
        self.m = 0

    def nombre_sommets(self):
        return self.n

    def nombre_aretes(self):
        return self.m
        
    def ajout_sommet(self, classe_sommet, label = ""):
        nouveau_sommet = classe_sommet(self.n, label)
        self.sommets[self.n] = nouveau_sommet
        self.n += 1

    def ajout_arete(self, id_sommet_1, id_sommet_2):
        self.sommets[id_sommet_1].ajout_voisin(id_sommet_2)
        self.sommets[id_sommet_2].ajout_voisin(id_sommet_1)
        self.m += 2 

    def ajout_arete_ponderee(self, id_sommet_1, id_sommet_2, poids):
        self.sommets[id_sommet_1].ajout_voisin({'id':id_sommet_2, 'poids':poids})
        self.sommets[id_sommet_2].ajout_voisin({'id':id_sommet_1, 'poids':poids})
        self.m += 2

    def ajout_arete_digraphe(self, id_sommet_1, id_sommet_2, arete_de_1_vers_2, arete_de_2_vers_1):
        if arete_de_1_vers_2:
            self.sommets[id_sommet_1].ajout_voisin(id_sommet_2)
            self.m += 1
        if arete_de_2_vers_1:
            self.sommets[id_sommet_2].ajout_voisin(id_sommet_1)
            self.m += 1
    
    def voisinage_sommet(self, id_sommet):
        return [id_voisin for id_voisin in self.sommets[id_sommet].voisins]

    def __str__(self):
        if self.n == 0:
            return "Le graphe est vide."
        
        resultat = "Le graphe est composé des sommets :\n"
        
        for id in self.sommets:
            s = self.sommets[id]
            resultat += "-\tNuméro " + str(s.id)
            if s.label != "":
                resultat += "(label: " + s.label + ")"
            resultat += ", ayant pour voisin(s) le(s) sommet(s) numéro(s) : " + str(s.voisins) + "\n"
        resultat += "Au total, " + str(self.n) + " sommets et " + str(self.m) + " arêtes."
        return resultat
    
    def __getitem__(self, id):
        return self.sommets[id]

## Rappel du code de Tas-Min et ajout de l'extrait du minimum

In [47]:
import math

def gauche(i):
    return 2*i+1

def droite(i):
    return 2*i+2

def entasser_min(A, i):
    l = gauche(i)
    r = droite(i)
    
    if l < len(A) and A[l] < A[i]:
        min = l
    else:
        min = i

    if r < len(A) and A[r] < A[min]:
        min = r

    if min != i:
        A[i], A[min] = A[min], A[i]
        A = entasser_min(A, min)
    return A

def construction_tas_min(A):
    for i in range(math.floor(len(A)/2) - 1, -1, -1):
        A = entasser_min(A, i)
    return A

In [48]:
def extract_min(A):
    if len(A) == 0:
        return 
    min_value = A[0]
    A[0] = A[len(A) - 1]
    entasser_min(A, 0)
    
    A = A[:-1]
    return min_value, A

In [49]:
def remonter_valeur(A, valeur):
    '''
    Cette fonction permet de remonter un sommet dans la file de priorité lorsque l'attribut distance de ce dernier est modifié.
    A est un Tas-Min représenté par une liste Python contenant des objets SommetDijkstra,
    valeur est l'objet SommetDijkstra à remonter dans la file de priorité.
    '''
    idx_valeur = A.index(valeur)
    while idx_valeur >= 0 and A[idx_valeur] < A[idx_valeur // 2]:
        tmp = A[idx_valeur]
        A[idx_valeur] = A[idx_valeur // 2]
        A[idx_valeur // 2] = tmp 

        idx_valeur = idx_valeur // 2
    return A

## Algorithme de Dijkstra

In [50]:
class SommetDijkstra(Sommet):
    def __init__(self, id, label):
        super().__init__(id, label)
        self.distance = math.inf 
        self.parent = None

    def __lt__(self, other):
        return self.distance < other.distance  

In [51]:
def dijkstra(graphe, s):
    S = set()
    Q = []
    
    for sommet in graphe.sommets:
        graphe[sommet].distance = math.inf
        graphe[sommet].parent = None
        Q.append(graphe[sommet])
    
    graphe[s].distance = 0
    Q = construction_tas_min(Q)
    
    while len(Q) > 0:
        u, Q = extract_min(Q)
        S.add(u.id)

        for v in graphe[u.id].voisins:
            v_sommet_id, v_poids = v['id'], v['poids']

            nouvelle_distance = u.distance + v_poids
            if not v_sommet_id in S and graphe[v_sommet_id].distance > nouvelle_distance:
                graphe[v_sommet_id].distance = nouvelle_distance
                Q = remonter_valeur(Q, graphe[v_sommet_id])
                graphe[v_sommet_id].parent = u.id
    return graphe

In [52]:
graphe = Graphe()

for i in range(11):
    graphe.ajout_sommet(SommetDijkstra)

import random

for i in range(11):
    for j in range(11):
        if i != j:
            graphe.ajout_arete_ponderee(i, j, random.randint(1, 40))

In [53]:
graphe = Graphe()

for i in range(0, 11):
    graphe.ajout_sommet(SommetDijkstra)

graphe.ajout_arete_ponderee(0, 1, 14)
graphe.ajout_arete_ponderee(0, 2, 3)
graphe.ajout_arete_ponderee(0, 7, 5)
graphe.ajout_arete_ponderee(1, 2, 5)
graphe.ajout_arete_ponderee(2, 4, 2)
graphe.ajout_arete_ponderee(4, 7, 9)
graphe.ajout_arete_ponderee(4, 9, 11)
graphe.ajout_arete_ponderee(9, 10, 7)
graphe.ajout_arete_ponderee(7, 10, 2)
graphe.ajout_arete_ponderee(1, 3, 19)
graphe.ajout_arete_ponderee(3, 5, 7)
graphe.ajout_arete_ponderee(5, 4, 3)
graphe.ajout_arete_ponderee(2, 8, 1)
graphe.ajout_arete_ponderee(1, 6, 20)
graphe.ajout_arete_ponderee(6, 8, 2)

In [54]:
print(graphe)

Le graphe est composé des sommets :
-	Numéro 0, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 1, 'poids': 14}, {'id': 2, 'poids': 3}, {'id': 7, 'poids': 5}]
-	Numéro 1, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 0, 'poids': 14}, {'id': 2, 'poids': 5}, {'id': 3, 'poids': 19}, {'id': 6, 'poids': 20}]
-	Numéro 2, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 0, 'poids': 3}, {'id': 1, 'poids': 5}, {'id': 4, 'poids': 2}, {'id': 8, 'poids': 1}]
-	Numéro 3, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 1, 'poids': 19}, {'id': 5, 'poids': 7}]
-	Numéro 4, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 2, 'poids': 2}, {'id': 7, 'poids': 9}, {'id': 9, 'poids': 11}, {'id': 5, 'poids': 3}]
-	Numéro 5, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 3, 'poids': 7}, {'id': 4, 'poids': 3}]
-	Numéro 6, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [{'id': 1, 'poids': 20}, {'id': 8, 'poids': 2}]
-	Numéro 7, ayant pour voisin(s) le(s) s

In [55]:
debut = 5
fin = 10

graphe = dijkstra(graphe, debut)

sommet = graphe[fin]
liste_sommets = []
distance_totale = 0

while sommet.id != debut:
    liste_sommets.insert(0, sommet.id)
    distance_totale += sommet.distance
    sommet = graphe[sommet.parent]
liste_sommets.insert(0, debut)

print("Liste des sommets du plus court chemin de " + str(debut) + " à " + str(fin) + " : " + str(liste_sommets))
print("Longueur du chemin : " + str(distance_totale)) 

Liste des sommets du plus court chemin de 5 à 10 : [5, 4, 7, 10]
Longueur du chemin : 29
